# A1.1 · Architecture review when the system acts

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

---

**Risk.** PDF threat models go stale the moment the agent's tools change.

**Control.** Living, continuously re-evaluated threat models over the three planes separately.

**This lab.** Make the threat model a living artefact that moves when the tools move.

| | |
|---|---|
| Open-source tooling | OWASP Threat Dragon, kagent |
| Open-weight models | GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A1.1"))

A threat model goes stale the moment the tool manifest changes — and adding a tool is not a code change, so nothing in your change process notices. The useful artefact is therefore not the model. It is the *diff*.

In [ ]:
from cybercommons import planes
W = planes.Tool

v1 = planes.Manifest("review-agent", [
    W("read_file"), W("search_code"),
    W("post_comment", writes=True, scope="project"),
], approval_required=set(), rung="L2.5")

v2 = planes.Manifest("review-agent", [
    W("read_file"), W("search_code"),
    W("post_comment", writes=True, scope="project"),
    W("merge_pr", writes=True, scope="project", reversible=False),   # added Tuesday
], approval_required=set(), rung="L2.5")

d = planes.diff_manifests(v1, v2)
print("added:  ", d["added"])
print("blast:  ", d["blast_before"], "→", d["blast_after"], f"(delta {d['delta']:+d})")
for p in d["new_problems"]:
    print("new problem:", p)

One line in a config file doubled the blast radius and introduced an irreversible action with no gate. No pull request touched the agent's code. This is why the review has to run against the manifest, continuously, and not against a document written at design time.

### Expect

`merge_pr` appears in `added`, the blast radius roughly doubles, and a new problem is reported: an irreversible ungated tool below L3.

### Your turn

Wire `diff_manifests` into a check that runs whenever the manifest file changes and fails when `delta > 0` without a matching entry in `approval_required`. That is a living threat model in ~10 lines.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A1.1.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*